### Building Chatbot

here we will learn how to build and implement an LLM powered Chatbot. This chatbot will able to have conversation and remember previous interaction.

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain_groq import ChatGroq
model=ChatGroq(model="openai/gpt-oss-safeguard-20b")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.0', 'langchain': '1.3.15'}}, profile={'name': 'Safety GPT OSS 20B', 'release_date': '2025-03-05', 'last_updated': '2025-03-05', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001F77FE26560>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001F77FE26470>, model_name='openai/gpt-oss-safeguard-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [3]:
from langchain_core.messages import HumanMessage
response=model.invoke([HumanMessage(content="Hi, My name is Anisha. How are you?")])
response

AIMessage(content='Hi Anisha! 👋 I’m doing great—thanks for asking. How about you? Anything interesting happening today?', additional_kwargs={'reasoning_content': 'We need to respond as ChatGPT. The user says "Hi, My name is Anisha. How are you?" We should respond in a friendly manner, ask a question, maybe. Let\'s produce a response.'}, response_metadata={'token_usage': {'completion_tokens': 77, 'prompt_tokens': 83, 'total_tokens': 160, 'completion_time': 0.075489944, 'completion_tokens_details': {'reasoning_tokens': 44}, 'prompt_time': 0.005392541, 'prompt_tokens_details': None, 'queue_time': 0.047057769, 'total_time': 0.080882485}, 'model_name': 'openai/gpt-oss-safeguard-20b', 'system_fingerprint': 'fp_b5ae46a825', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0423b-2357-7a13-9027-f5bcd5c2fcff-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 83, 'output_tokens': 77, 'total_tokens': 160, 'outp

In [5]:
from langchain_core.messages import AIMessage
result=model.invoke(
    [
        HumanMessage(content="Hi, My name is Anisha. How are you?"),
        AIMessage(content="Hi Anisha! 👋 I’m doing great—thanks for asking. How about you? Anything interesting happening today?"),
        HumanMessage(content="Can you tell me my name?")
    ]
)

In [6]:
result

AIMessage(content='Sure! Your name is Anisha. If you’d like me to remember it for later in this conversation, just let me know!', additional_kwargs={'reasoning_content': 'User: "Can you tell me my name?" They just told their name: Anisha. So answer: "Your name is Anisha." Also maybe confirm? But need to be polite.'}, response_metadata={'token_usage': {'completion_tokens': 76, 'prompt_tokens': 124, 'total_tokens': 200, 'completion_time': 0.074668171, 'completion_tokens_details': {'reasoning_tokens': 40}, 'prompt_time': 0.006855671, 'prompt_tokens_details': None, 'queue_time': 0.047712489, 'total_time': 0.081523842}, 'model_name': 'openai/gpt-oss-safeguard-20b', 'system_fingerprint': 'fp_b5ae46a825', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0423b-85e2-7ff0-ba3c-e0098977cd5f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 124, 'output_tokens': 76, 'total_tokens': 200, 'output_token_details':

### Message History
we can use message history class to wrap our model and make it stateful. 
This will keep track of input and output of the model, and store them in some datastore. 
Further interactions will then load those messages and pass them into chain as the part of the input.
Lets see how to use this.

In [18]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store={}

def get_session_history(session_id:str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id]=ChatMessageHistory()
    return store[session_id]

with_message_history=RunnableWithMessageHistory(model,get_session_history)

In [19]:
config={"configurable":{"session_id":"chat1"}}

In [20]:
result=with_message_history.invoke(
    [HumanMessage(content="I am Anisha and I am learning AIML")],
    config=config
)

In [21]:
result.content

'Hi Anisha! 👋🏼\n\nIt’s great that you’re diving into AIML (Artificial Intelligence Markup Language). Whether you’re just starting out or already working on a bot, I’m here to help. Here are a few ways we can get going:\n\n| Topic | What it covers | Quick tip |\n|-------|----------------|-----------|\n| **Basics** | Structure of an AIML file, `<category>`, `<pattern>`, `<template>` | Start with a simple “Hello” bot to see how patterns match. |\n| **Advanced patterns** | Wildcards (`*`, `_`), regex, and the `that`/`topic` tags | Use `*` to capture anything and then process it in the template. |\n| **Response templates** | Variables, `<srai>`, `<random>`, `<think>` | `<srai>` lets you reuse patterns; great for synonyms. |\n| **Bot behavior** | Memory (`<set>`/`<get>`), context, and session handling | Keep track of user state with variables. |\n| **Integration** | Connecting AIML to a web server, chat UI, or other languages | Many frameworks (e.g., Program\u202fAI, MyBot) let you plug in A

In [22]:
with_message_history.invoke(
    [HumanMessage(content="What is my name and what i am learning?")],
    config=config
)

AIMessage(content='You’re **Anisha**, and you’re learning **AIML (Artificial Intelligence Markup Language)**.', additional_kwargs={'reasoning_content': 'The user says "I am Anisha and I am learning AIML". They ask: "What is my name and what i am learning?" So the answer should be: Name is Anisha, learning AIML. Also maybe confirm. Should respond.'}, response_metadata={'token_usage': {'completion_tokens': 83, 'prompt_tokens': 730, 'total_tokens': 813, 'completion_time': 0.082842121, 'completion_tokens_details': {'reasoning_tokens': 52}, 'prompt_time': 0.052054074, 'prompt_tokens_details': None, 'queue_time': 0.098079134, 'total_time': 0.134896195}, 'model_name': 'openai/gpt-oss-safeguard-20b', 'system_fingerprint': 'fp_b5ae46a825', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a04288-4066-7882-99ed-5e207839d6dd-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 730, 'output_tokens': 83, 'total_toke

In [23]:
""" 
Chane the config----> Session ID ----> 
By this we will use new session with some different session ID and 
model will not have access to another session chat
"""
config={"configurable":{"session_id":"chat2"}}
with_message_history.invoke(
    [HumanMessage(content="What is my name and what i am learning?")],
    config=config
)

AIMessage(content='I’m not sure what your name is or what you’re studying—could you let me know? Once I have that, I’ll gladly help you with whatever you’re learning!', additional_kwargs={'reasoning_content': 'The user asks: "What is my name and what i am learning?" We don\'t know their name or what they are learning. This is a trick? Possibly they want to see if the assistant can guess? But we can\'t know. Maybe we should respond with a polite request for clarification. Or maybe it\'s a test of memory? The user hasn\'t provided name or subject. So respond asking for details. Also, maybe we can say we don\'t have that information. Also, we can say we cannot know. Provide a friendly answer. Let\'s do that.'}, response_metadata={'token_usage': {'completion_tokens': 156, 'prompt_tokens': 81, 'total_tokens': 237, 'completion_time': 0.153523893, 'completion_tokens_details': {'reasoning_tokens': 111}, 'prompt_time': 0.003796617, 'prompt_tokens_details': None, 'queue_time': 0.047139803, 'tota

In [29]:
with_message_history.invoke(
    [HumanMessage(content="Hey, My name is Stela.")],
    config=config
)

AIMessage(content='Got it—your name is Stela! If there’s anything you’d like to chat about or need help with, just let me know.', additional_kwargs={'reasoning_content': "We need to respond. Probably just respond acknowledging name. Let's do friendly."}, response_metadata={'token_usage': {'completion_tokens': 54, 'prompt_tokens': 244, 'total_tokens': 298, 'completion_time': 0.05529638, 'completion_tokens_details': {'reasoning_tokens': 16}, 'prompt_time': 0.011859483, 'prompt_tokens_details': None, 'queue_time': 0.047190906, 'total_time': 0.067155863}, 'model_name': 'openai/gpt-oss-safeguard-20b', 'system_fingerprint': 'fp_b5ae46a825', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a04291-1d3a-7632-898e-e52927108683-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 244, 'output_tokens': 54, 'total_tokens': 298, 'output_token_details': {'reasoning': 16}})

In [30]:
with_message_history.invoke(
    [HumanMessage(content="What is my name?")],
    config=config
)

AIMessage(content='You’re Stela.', additional_kwargs={'reasoning_content': 'We need to respond that the user says name is Stela. So answer: Your name is Stela. Probably just repeat.'}, response_metadata={'token_usage': {'completion_tokens': 41, 'prompt_tokens': 288, 'total_tokens': 329, 'completion_time': 0.040376851, 'completion_tokens_details': {'reasoning_tokens': 27}, 'prompt_time': 0.015250795, 'prompt_tokens_details': None, 'queue_time': 0.052940544, 'total_time': 0.055627646}, 'model_name': 'openai/gpt-oss-safeguard-20b', 'system_fingerprint': 'fp_b5ae46a825', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a04291-2df8-7981-a93b-ef722bc7d214-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 288, 'output_tokens': 41, 'total_tokens': 329, 'output_token_details': {'reasoning': 27}})